# Week 1: Diffusion Policy on PushT

Settings: Accelerator = GPU T4, Internet = On, Secret `WANDB_API_KEY` attached.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["MUJOCO_GL"] = "egl"
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ.pop("PYTHONPATH", None)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q uv
!uv pip install --system "lerobot[pusht,training]"
!python -c "import sys, torch, lerobot; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available())"

## Full run
80k steps x batch 32 at ~3.6 steps/s on a T4 is ~6.2 h plus two 50-episode evals. The step-80k eval is the result.

In [ ]:
STEPS = 80000
!lerobot-train --policy.type=diffusion --dataset.repo_id=lerobot/pusht --env.type=pusht \
  --steps={STEPS} --batch_size=32 --policy.use_amp=true --seed=0 \
  --save_freq=40000 --env_eval_freq=40000 --eval.n_episodes=50 --eval.batch_size=10 \
  --wandb.enable=true --wandb.disable_artifact=true --wandb.project=scratch --job_name=dp-pusht-80k-bs32-s0 \
  --policy.push_to_hub=false --output_dir=/kaggle/working/dp_pusht